In [61]:

import pandas as pd
import numpy as np
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
#from sklearn.cross_validation import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import re


In [62]:
# Data import for the use in th notebook
spam_data = pd.read_csv(
    "sms_scam_detection_dataset_merged_with_lang.csv",
    encoding="utf-8",
    low_memory=False
)
print(spam_data.columns)


Index(['label', 'text', 'URL', 'EMAIL', 'PHONE', 'lang', 'Unnamed: 6',
       'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10'],
      dtype='str')


In [63]:
# Exploration of the dataset
spam_data.head()

,label,text,URL,EMAIL,PHONE,lang,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
1,ham,What's up? Do you want me to come online? If y...,No,No,No,en,NaN,NaN,NaN,NaN,NaN
2,ham,So u workin overtime nigpun?,No,No,No,en,NaN,NaN,NaN,NaN,NaN
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,en,NaN,NaN,NaN,NaN,NaN
4,spam,Please Stay At Home. To encourage the notion o...,No,No,No,en,NaN,NaN,NaN,NaN,NaN


In [64]:
# As the column label contain only Ham and spam , we are gonna replace them by 1 and 0
spam_data= spam_data.replace(['ham','spam'],[0,1])


In [65]:
# Some column have NAN value
# We must suppress all the NAN columns
spam_data = spam_data.drop(columns=["Unnamed: 7","Unnamed: 8","Unnamed: 9","Unnamed: 10"])
spam_data = spam_data.drop(columns=["Unnamed: 6"])

In [66]:
spam_data = spam_data.rename(columns={
    "label": "Label",
    "text": "Text",
    "URL": "Url",
    "EMAIL": "Email",
    "PHONE": "Phone",
    "lang": "Language"
})
spam_data.head()

,Label,Text,Url,Email,Phone,Language
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,en
1,0,What's up? Do you want me to come online? If y...,No,No,No,en
2,0,So u workin overtime nigpun?,No,No,No,en
3,0,"Also sir, i sent you an email about how to log...",No,No,No,en
4,1,Please Stay At Home. To encourage the notion o...,No,No,No,en


In [67]:
# Let's treat the others columns
# column URL
print("Unique Value for the column URL")
spam_data["Url"].value_counts()


Unique Value for the column URL


Url
No     5744
yes     205
Name: count, dtype: int64

In [68]:
spam_data['Url'] = spam_data['Url'].replace({'Yes': 1, 'No': 0})

In [69]:
print("Unique Value for the column EMAIL")
spam_data['Email'].value_counts()



Unique Value for the column EMAIL


Email
No     5930
yes      19
Name: count, dtype: int64

In [70]:
spam_data['Email']=spam_data['Email'].replace({'Yes':1,'No':0})

In [71]:
print("Unique Value for the column PHONE")
spam_data["Phone"].value_counts()

Unique Value for the column PHONE


Phone
No     5115
yes     834
Name: count, dtype: int64

In [72]:
spam_data['Phone']=spam_data['Phone'].replace({'Yes':1,'No':0})

In [73]:
print("Unique Value for the column lang")
spam_data['Language'].value_counts()

Unique Value for the column lang


Language
en         5850
hindi      5129
french     5108
german     5108
english     275
und          13
hi            8
ml            8
nl            7
no            6
kn            5
ro            5
kha           4
ta            4
ru            3
bn            3
es            3
te            3
ar            2
pt            2
lb            2
zh            2
it            1
af            1
fil           1
ms            1
sv            1
fr            1
ku            1
ht            1
aa            1
sw            1
ny            1
tr            1
hr            1
vi            1
pl            1
kri           1
gu            1
fy            1
sm            1
Name: count, dtype: int64

In [74]:
# The column for languages contains the language according the ISO 931 format
# We should try to normalize or simplify the use of this column
mapping = {
    # Déjà sous forme de codes ISO 639-1
    "en": "english",
    "fr": "french",
    "de": "german",
    "hi": "hindi",
    "ml": "malayalam",
    "nl": "dutch",
    "no": "norwegian",
    "kn": "kannada",
    "ro": "romanian",
    "ta": "tamil",
    "ru": "russian",
    "bn": "bengali",
    "es": "spanish",
    "te": "telugu",
    "ar": "arabic",
    "pt": "portuguese",
    "zh": "chinese",
    "it": "italian",
    "af": "afrikaans",
    "ms": "malay",
    "sv": "swedish",
    "ku": "kurdish",
    "ht": "haitian creole",
    "sw": "swahili",
    "ny": "chichewa",
    "tr": "turkish",
    "hr": "croatian",
    "vi": "vietnamese",
    "pl": "polish",
    "gu": "gujarati",
    "fy": "western frisian",
    "sm": "samoan",
    "aa": "afar",

    # Codes ISO 639-3
    "fil": "filipino",
    "kha": "khasi",
    "kri": "krio",

    # Cas particulier
    "und": "unknown"
}

In [75]:
spam_data['Language'] = spam_data['Language'].replace(mapping)

In [76]:
spam_data['Language'].value_counts()

Language
english            6125
hindi              5137
french             5109
german             5108
unknown              13
malayalam             8
dutch                 7
norwegian             6
kannada               5
romanian              5
khasi                 4
tamil                 4
russian               3
bengali               3
spanish               3
telugu                3
arabic                2
portuguese            2
lb                    2
chinese               2
italian               1
afrikaans             1
filipino              1
malay                 1
swedish               1
kurdish               1
haitian creole        1
afar                  1
swahili               1
chichewa              1
turkish               1
croatian              1
vietnamese            1
polish                1
krio                  1
gujarati              1
western frisian       1
samoan                1
Name: count, dtype: int64

In [77]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english
1,0,What's up? Do you want me to come online? If y...,0,0,0,english
2,0,So u workin overtime nigpun?,0,0,0,english
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english


In [78]:
spam_data.dtypes

Label       object
Text        object
Url         object
Email       object
Phone       object
Language       str
dtype: object

In [79]:
# Let's treat the column Test with NLP
# 1 - Count the number of word in each text
spam_data['Word_Count'] = 0
for i in np.arange(0, len(spam_data.Text)):
    spam_data.loc[i, 'Word_Count'] = len(str(spam_data.loc[i, 'Text']).split())

In [80]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [81]:
corpus = []
ps = PorterStemmer()

In [82]:
# Some original messages
print (spam_data["Text"][0])
spam_data["Text"][1]

Your opinion about me? 1. Over 2. Jada 3. Kusruthi 4. Lovable 5. Silent 6. Spl character 7. Not matured 8. Stylish 9. Simple Pls reply..


"What's up? Do you want me to come online? If you are free we can talk sometime�"

In [83]:
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [84]:
# Translator to translate all text in english to normalize the dataset
from deep_translator import  GoogleTranslator
def translate_sentence(sentence):
    traducteur = GoogleTranslator(source="auto",destination="en")
    translated_sentence = traducteur.translate(sentence)
    return translated_sentence

In [85]:
# Let's translate all languages others than english
spam_data = spam_data[spam_data["Language"] == "english"]
# to suppress the old index that are NAN now
spam_data = spam_data.reset_index(drop=True)

In [86]:
# Let's print the new dataset
spam_data.head()

,Label,Text,Url,Email,Phone,Language,Word_Count
0,0,Your opinion about me? 1. Over 2. Jada 3. Kusr...,0,0,0,english,26
1,0,What's up? Do you want me to come online? If y...,0,0,0,english,17
2,0,So u workin overtime nigpun?,0,0,0,english,5
3,0,"Also sir, i sent you an email about how to log...",0,0,0,english,33
4,1,Please Stay At Home. To encourage the notion o...,0,0,0,english,24


In [87]:

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jemnis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [88]:
corpus = []

for i in range(len(spam_data)):

    msg = str(spam_data['Text'][i])

    # Regex
    msg = re.sub(r'\b[\w\-.]+?@\w+?\.\w{2,4}\b', 'emailaddr', msg)
    msg = re.sub(r'(http[s]?\S+)|(\w+\.[A-Za-z]{2,4}\S*)', 'httpaddr', msg)
    msg = re.sub(r'£|\$', 'moneysymb', msg)
    msg = re.sub(r'\b(\+\d{1,2}\s)?\d?[\-(.]?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b', 'phonenumbr', msg)
    msg = re.sub(r'\d+(\.\d+)?', 'numbr', msg)

    # Remove punctuation
    msg = re.sub(r'[^\w\d\s]', ' ', msg)

    # Lowercase
    msg = msg.lower()

    # Tokenize
    words = msg.split()

    # Stopwords + stemming (anglais seulement)
    words = [
        ps.stem(word)
        for word in words
        if word not in set(stopwords.words('english'))
    ]

    msg = ' '.join(words)

    corpus.append(msg)

In [89]:
cv = CountVectorizer()
x = cv.fit_transform(corpus).toarray()
